# Issue #78: fixed-manifest Qwen and Hashformers benchmark

This notebook reproduces the corrected protocol-v3 unquantized FP16 Qwen comparison and the fixed-manifest Hashformers baselines on a Google Colab Tesla T4. Protocol v3 separates strict insertion-only validity from scored segmentation proposals: it deterministically projects recoverable boundaries onto the original source and otherwise records an unchanged-input fallback. The Hashformers runs use the PR #80 adaptive candidate controller with `gpu_batch_size="auto"`. The notebook installs Hashformers directly from each checked-out Git revision used by the recorded runs, keeps generated artifacts outside the checkout so provenance remains clean, and runs each model in a separate process. Start with a fresh GPU runtime.

In [ ]:
!git clone https://github.com/ruanchaves/hashformers.git /content/hashformers
%cd /content/hashformers
!git checkout 59910585795306ca68aefeeba50b30827ae27d12
!python -m pip install -q --upgrade 'transformers>=4.51,<6' 'accelerate>=1'
!python -m pip install -q --no-deps -e /content/hashformers
!nvidia-smi

import subprocess

assert not subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd='/content/hashformers', text=True
).strip(), 'Benchmark checkout must be clean'

In [ ]:
!python scripts/qwen_benchmark.py run \
  --model qwen3 \
  --manifest benchmarks/qwen/samples.jsonl \
  --device cuda:0 \
  --precision float16 \
  --quantization none \
  --warmup 5 \
  --max-new-tokens 64 \
  --output-dir /content/qwen-v3-results/qwen3

In [ ]:
!python scripts/qwen_benchmark.py run \
  --model qwen2-historical \
  --manifest benchmarks/qwen/samples.jsonl \
  --device cuda:0 \
  --precision float16 \
  --quantization none \
  --warmup 5 \
  --max-new-tokens 64 \
  --output-dir /content/qwen-v3-results/qwen2

In [ ]:
!python scripts/qwen_benchmark.py summarize \
  --predictions /content/qwen-v3-results/qwen3/predictions.jsonl \
                /content/qwen-v3-results/qwen2/predictions.jsonl \
  --output /content/qwen-v3-results/comparison.json

In [ ]:
import json
from pathlib import Path

comparison = json.loads(
    Path('/content/qwen-v3-results/comparison.json').read_text()
)
for run in comparison['runs']:
    print(
        run['model_label'],
        run['overall']['accuracy'],
        run['overall']['strict_output_accuracy'],
        run['overall']['invalid_output_rate'],
        run['overall']['recovered_prediction_rate'],
        run['overall']['source_fallback_rate'],
        run['overall']['output_wrapper_rate'],
    )
print(comparison['paired_comparisons'])

In [ ]:
!git checkout d4180e11e383608387685d8f595103adfae8ee72
!python -m pip install -q -e /content/hashformers

import subprocess

assert not subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd='/content/hashformers', text=True
).strip(), 'Benchmark checkout must be clean'

In [ ]:
!python scripts/hashformers_benchmark.py run \
  --model gpt2 --device cuda:0 \
  --gpu-batch-size auto --max-gpu-batch-size 512 \
  --output-dir /content/qwen-v3-results/hashformers-gpt2

In [ ]:
!python scripts/hashformers_benchmark.py run \
  --model distilgpt2 --device cuda:0 \
  --gpu-batch-size auto --max-gpu-batch-size 512 \
  --output-dir /content/qwen-v3-results/hashformers-distilgpt2

In [ ]:
!python scripts/hashformers_benchmark.py run \
  --model rugpt3small --device cuda:0 \
  --gpu-batch-size auto --max-gpu-batch-size 512 \
  --output-dir /content/qwen-v3-results/hashformers-rugpt3small

In [ ]:
!python scripts/hashformers_benchmark.py compare \
  --predictions /content/qwen-v3-results/qwen3/predictions.jsonl \
                /content/qwen-v3-results/qwen2/predictions.jsonl \
                /content/qwen-v3-results/hashformers-gpt2/predictions.jsonl \
                /content/qwen-v3-results/hashformers-distilgpt2/predictions.jsonl \
                /content/qwen-v3-results/hashformers-rugpt3small/predictions.jsonl \
  --output /content/qwen-v3-results/combined_comparison.json

In [ ]:
for model in ('gpt2', 'distilgpt2', 'rugpt3small'):
    metadata = json.loads(
        Path(f'/content/qwen-v3-results/hashformers-{model}/run_metadata.json').read_text()
    )
    print(
        model,
        metadata['results']['overall']['accuracy'],
        metadata['segmentation']['gpu_batch_size'],
        metadata['measurement']['candidate_batch_telemetry'],
    )
combined = json.loads(
    Path('/content/qwen-v3-results/combined_comparison.json').read_text()
)
print(combined['paired_comparisons'])

In [ ]:
!cd /content/hashformers && git status --porcelain && git rev-parse HEAD && python -m pytest tests/test_segmenter.py -q

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    '/content/hashformers-issue-78-fixed-manifest-results',
    'zip',
    '/content/qwen-v3-results',
)
files.download(archive)